In [1]:
# [1] minimal state from the live instrument — read-only, no motion
from types import SimpleNamespace

from spm_agent.mcp.spm_session import call
from spm_agent.schemas.instrument import ScanSettings

raw = await call("pfm_get_experiment_status")
ss = ScanSettings(
    x_scan_center_m=raw["x_scan_center_m"],
    y_scan_center_m=raw["y_scan_center_m"],
    scan_size_m=raw["scan_size_m"],
    pixels=raw["n_points"],
    scan_rate_hz=raw["scan_rate_hz"],
)
print(ss)
print(f"pixel size: {ss.scan_size_m / ss.pixels * 1e9:.1f} nm   feedback_on: {raw['feedback_on']}")

x_scan_center_m=0.0 y_scan_center_m=0.0 scan_size_m=9.9999997e-06 pixels=256 scan_rate_hz=0.8979885
pixel size: 39.1 nm   feedback_on: False


In [3]:
# [2] fabricate exactly what move_tip_node reads, then move
from spm_agent.config import IMAGE_ROW0_AT_TOP
from spm_agent.nodes.move_tip_node import move_tip_node
from spm_agent.utils.pick_utils import pixel_to_frame_xy

ROW, COL = 0, 128          # x at the centre, y well off-centre -> reveals the row convention

x, y = pixel_to_frame_xy(ss, ROW, COL, IMAGE_ROW0_AT_TOP)
print(f"px({ROW},{COL}) -> ({x*1e6:+.4f}, {y*1e6:+.4f}) um")

state = {
    "pending": {"kind": "loop", "decision_index": 0, "pixel_yx": (ROW, COL)},
    "experimental_records": [
        {"kind": "image", "scan_index": 0,
         "instrument_params": SimpleNamespace(scan_settings=ss)},   # only .scan_settings is read
    ],
    # scanner_calibrations omitted on purpose -> the tool parks at the frame centre
    # first ("slower, always correct"). Add it to test the direct path.
}

await move_tip_node(state)

px(0,128) -> (+0.0195, +4.9805) um
[move_tip] px(0,128) -> (+0.0195, +4.9805) um on a 10.00 um frame (39.1 nm/px)


RuntimeError: Z feedback is OFF after the move — the tip is not in contact and a hysteresis loop cannot be measured. Engage before continuing.

In [1]:
# [1] state from the last run + live instrument — read-only, no motion
import json
from types import SimpleNamespace as NS

from spm_agent.config import RUNS_ROOT, new_run
from spm_agent.mcp.spm_session import call
from spm_agent.schemas.experimental_decision import ExperimentDecision
from spm_agent.schemas.instrument import (InstrumentState, ScanSettings, LoopSettings,
                                          ProbePosition, PFMExcitation, ContactFeedback)

src     = sorted(p for p in RUNS_ROOT.iterdir() if p.is_dir())[-1]
dec_dir = sorted((src / "decisions").glob("decision_*"))[-1]
decision = ExperimentDecision(**json.loads((dec_dir / "decision.json").read_text()))
pick     = json.loads((dec_dir / "pick.json").read_text())
print(f"{src.name} | {dec_dir.name} | {decision.target_criterion} / "
      f"{decision.target_strategy} | px {pick['pixel_yx']}")

raw = await call("pfm_get_experiment_status")
live = InstrumentState(
    scan_settings=ScanSettings(
        x_scan_center_m=raw["x_scan_center_m"], y_scan_center_m=raw["y_scan_center_m"],
        scan_size_m=raw["scan_size_m"], pixels=raw["n_points"],
        scan_rate_hz=raw["scan_rate_hz"]),
    loop_settings=LoopSettings(
        v_dc_max_v=raw["v_dc_max"], frequency_hz=raw["loop_frequency"],
        var0_phase=raw["var0_loop_phase"], var1_pulsetime_s=raw["var1_pulsetime_s"],
        n_cycles=raw["n_cycles"]),
    probe_position=ProbePosition(x_m=0.0, y_m=0.0),        # not read by loop_plan
    pfm_excitation=PFMExcitation(
        drive_amplitude_v=raw["v_ac_v"], drive_frequency_hz=raw["f_dart_hz"],
        dart_igain=raw["dart_igain"], dart_width_hz=raw["f_dart_width_hz"]),
    contact_feedback=ContactFeedback(setpoint_v=raw["setpoint_defl_v"], gain=raw["igain"]))
print(live.loop_settings)

state = {
    "experiment_tasks":   ["Compare switching inside domains vs near domain walls"],
    "experiment_context": "DART PFM: two drive frequencies straddle the contact resonance.",
    "instrument_state":   live,
    "decision_records":   [decision],
    "experimental_records": [{"kind": "image", "scan_index": 0, "instrument_params": live}],
    "pending": {"kind": "loop",
                "decision_index": int(dec_dir.name.split("_")[1]),
                "pixel_yx": tuple(pick["pixel_yx"])},
}
print(new_run("loop_plan_test"))          # artifacts land here, not in the source run

20260817_185202_pick_point_single_1 | decision_01 | domain_interior_score / max | px [12, 242]
v_dc_max_v=1.5 frequency_hz=0.30048078 var0_phase=0.5 var1_pulsetime_s=0.050000001 n_cycles=2
C:\Users\Asylum User\Documents\GitHub\SPM_agent_simple\src\runs\20260818_151336_loop_plan_test


In [2]:
# [2] A: first loop -> deterministic guard (no LLM).  B: after an 'unsaturated' loop -> LLM
from spm_agent.nodes.loop_plan_node import loop_plan_node
from spm_agent.schemas.loop_plan import loop_plan_diff

out_a = await loop_plan_node(state)
plan_a = out_a["pending"]["params"]
print("A:", plan_a.diagnosis, "|", plan_a.loop_settings, "\n")

prev = {"kind": "loop", "scan_index": 0, "pixel_yx": tuple(pick["pixel_yx"]),
        "decision_index": state["pending"]["decision_index"],
        "instrument_params": live,
        "requested_params": plan_a,
        "loop_review": NS(is_ferroelectric_like=True, loop_quality="unsaturated",
                          issues=["branches do not flatten at +/- Vmax",
                                  "loop height small relative to branch noise"],
                          interpretation="Switching is present but the sweep never saturates."),
        "loop_params": {"off_field": NS(v_c_rising=0.62, v_c_falling=-0.55,
                                        loop_width_v=1.17, imprint_v=0.03,
                                        loop_height_m=8.0e-11)}}

state_b = {**state, "experimental_records": state["experimental_records"] + [prev]}
out_b = await loop_plan_node(state_b)
plan_b = out_b["pending"]["params"]

print("B diagnosis:", plan_b.diagnosis)
print("B reasoning:", plan_b.reasoning)
print("B changes  :", loop_plan_diff(plan_b, live) or "none")
print("B waveform :", plan_b.loop_settings)

[loop_plan] first loop — operator waveform adopted (v_dc_max 1.50 V, 2 cycles)
A: (deterministic guard) first loop of the session | v_dc_max_v=1.5 frequency_hz=0.30048078 var0_phase=0.5 var1_pulsetime_s=0.050000001 n_cycles=2 

[loop_plan] Loop 0.1 at this domain-interior site shows an unsaturated ferroelectric response: the branches do not flatten at ±1.50 V and the loop height is small relative to branch noise. The coercive voltages are modest (±0.6 V) and the loop width is ~1.17 V, indicating switching is occurring but the polarization has not been driven to saturation at the current bias amplitude.
[loop_plan] changes: {'v_dc_max_v': 3.0, 'frequency_hz': 0.3}
B diagnosis: Loop 0.1 at this domain-interior site shows an unsaturated ferroelectric response: the branches do not flatten at ±1.50 V and the loop height is small relative to branch noise. The coercive voltages are modest (±0.6 V) and the loop width is ~1.17 V, indicating switching is occurring but the polarization has not be

In [3]:
# [1] fresh calibration + frame + helpers
from spm_agent.config import new_run, IMAGE_ROW0_AT_TOP
from spm_agent.mcp.spm_session import call
from spm_agent.nodes.scanner_calibration_node import get_scanner_calibrations_node
from spm_agent.schemas.instrument import ScanSettings
from spm_agent.utils.pick_utils import pixel_to_frame_xy

print(new_run("move_tip_test"))

cal = (await get_scanner_calibrations_node({}))["scanner_calibrations"]   # MOVES tip to centre
print(cal)

raw = await call("pfm_get_experiment_status")
ss = ScanSettings(x_scan_center_m=raw["x_scan_center_m"], y_scan_center_m=raw["y_scan_center_m"],
                  scan_size_m=raw["scan_size_m"], pixels=raw["n_points"],
                  scan_rate_hz=raw["scan_rate_hz"])
px_m = ss.scan_size_m / ss.pixels
print(f"frame {ss.scan_size_m*1e6:.2f} um @ {ss.pixels} px ({px_m*1e9:.1f} nm/px), "
      f"centre ({ss.x_scan_center_m*1e6:+.4f}, {ss.y_scan_center_m*1e6:+.4f}) um | "
      f"feedback {raw['feedback_on']} | ROW0_AT_TOP={IMAGE_ROW0_AT_TOP}")


async def _lvdt():
    s = await call("pfm_get_experiment_status")
    return s["x_probe_lvdt_m"], s["y_probe_lvdt_m"]


async def move(x, y, use_offsets: bool):
    """Returns (tool payload, raw LVDT readback). LVDT is the physical truth —
    the tool's own x_m/y_m are derived from whatever offsets it used."""
    args = {"x_m": float(x), "y_m": float(y)}
    if use_offsets:
        args["x_scanner_offset_m"] = cal.x_scanner_offset_m
        args["y_scanner_offset_m"] = cal.y_scanner_offset_m
    d = await call("pfm_move_tip", args)
    return d, await _lvdt()

C:\Users\Asylum User\Documents\GitHub\SPM_agent_simple\src\runs\20260818_164258_move_tip_test
x_scanner_offset_m=-1.7972080222395838e-05 y_scanner_offset_m=1.5547959992315857e-05 x_lvdt_sens_m_per_v=9.2639584e-06 y_lvdt_sens_m_per_v=9.9031586e-06
frame 10.00 um @ 256 px (39.1 nm/px), centre (+0.0000, +0.0000) um | feedback False | ROW0_AT_TOP=True


In [6]:
# [2] TEST A — same target, two anchor paths. Decides whether the offsets are good.
ROW, COL = 64, 192                     # off-centre in both axes
tx, ty = pixel_to_frame_xy(ss, ROW, COL, IMAGE_ROW0_AT_TOP)
print(f"target px({ROW},{COL}) -> ({tx*1e6:+.4f}, {ty*1e6:+.4f}) um\n")

d1, l1 = await move(tx, ty, use_offsets=True)     # direct
d2, l2 = await move(tx, ty, use_offsets=False)    # tool re-anchors via frame centre

print(f"direct : reported ({d1['x_m']*1e6:+.4f}, {d1['y_m']*1e6:+.4f}) um | "
      f"lvdt ({l1[0]*1e6:+.4f}, {l1[1]*1e6:+.4f}) um")
print(f"slow   : reported ({d2['x_m']*1e6:+.4f}, {d2['y_m']*1e6:+.4f}) um | "
      f"lvdt ({l2[0]*1e6:+.4f}, {l2[1]*1e6:+.4f}) um")

dx, dy = (l1[0] - l2[0]), (l1[1] - l2[1])
print(f"\nLVDT delta: ({dx*1e9:+.1f}, {dy*1e9:+.1f}) nm = "
      f"({dx/px_m:+.2f}, {dy/px_m:+.2f}) px")
print("=> offsets OK" if max(abs(dx), abs(dy)) < 2 * px_m else
      "=> OFFSETS SUSPECT — set MOVE_USE_CALIBRATION_OFFSETS = False")

target px(64,192) -> (+2.5195, +2.4805) um

direct : reported (+2.5195, +2.4805) um | lvdt (-15.4525, +18.0284) um
slow   : reported (+2.5195, +2.4805) um | lvdt (-15.4525, +18.0284) um

LVDT delta: (+0.0, +0.0) nm = (+0.00, +0.00) px
=> offsets OK


In [8]:
# [3] TEST B, part 1 — top of the image under ROW0_AT_TOP=True
x, y = pixel_to_frame_xy(ss, 21, 235, IMAGE_ROW0_AT_TOP)
d, lv = await move(x, y, use_offsets=True)
# print(f"px(21,128) -> y {y*1e6:+.4f} um | reported {d['y_m']*1e6:+.4f} um")
# print("LOOK at the AR marker: it should sit near the TOP edge, horizontally centred.")

In [7]:
# [4] TEST B, part 2 — bottom of the image under ROW0_AT_TOP=True
x, y = pixel_to_frame_xy(ss, 234, 128, IMAGE_ROW0_AT_TOP)
d, lv = await move(x, y, use_offsets=False)
print(f"px(234,128) -> y {y*1e6:+.4f} um | reported {d['y_m']*1e6:+.4f} um")
print("LOOK again: the marker should now be near the BOTTOM edge.")
print("If it moved the other way, set IMAGE_ROW0_AT_TOP = False in config.py.")

px(234,128) -> y -4.1602 um | reported -4.1602 um
LOOK again: the marker should now be near the BOTTOM edge.
If it moved the other way, set IMAGE_ROW0_AT_TOP = False in config.py.


In [2]:
# [1] rebuild state from the last run + live instrument (read-only, no motion)
import json
from pathlib import Path

from spm_agent.config import RUNS_ROOT, new_run
from spm_agent.mcp.spm_session import call
from spm_agent.schemas.experimental_decision import ExperimentDecision
from spm_agent.schemas.importance_components import ComponentsMeta, SafetyMeta
from spm_agent.schemas.instrument import (InstrumentState, ScanSettings, LoopSettings,
                                          ProbePosition, PFMExcitation, ContactFeedback)

run = sorted((p for p in RUNS_ROOT.iterdir() if p.is_dir()),
             key=lambda p: p.stat().st_mtime)[-1]
dec = sorted((run / "decisions").glob("decision_*"))[-1]
decision = ExperimentDecision(**json.loads((dec / "decision.json").read_text()))
print(f"{run.name} | {dec.name} | {decision.target_criterion}/{decision.target_strategy}")

raw = await call("pfm_get_experiment_status")
live = InstrumentState(
    scan_settings=ScanSettings(
        x_scan_center_m=raw["x_scan_center_m"], y_scan_center_m=raw["y_scan_center_m"],
        scan_size_m=raw["scan_size_m"], pixels=raw["n_points"],
        scan_rate_hz=raw["scan_rate_hz"]),
    loop_settings=LoopSettings(
        v_dc_max_v=raw["v_dc_max"], frequency_hz=raw["loop_frequency"],
        var0_phase=raw["var0_loop_phase"], var1_pulsetime_s=raw["var1_pulsetime_s"],
        n_cycles=raw["n_cycles"]),
    probe_position=ProbePosition(x_m=0.0, y_m=0.0),
    pfm_excitation=PFMExcitation(
        drive_amplitude_v=raw["v_ac_v"], drive_frequency_hz=raw["f_dart_hz"],
        dart_igain=raw["dart_igain"], dart_width_hz=raw["f_dart_width_hz"]),
    contact_feedback=ContactFeedback(setpoint_v=raw["setpoint_defl_v"], gain=raw["igain"]))

# the scan record, rehydrated: save_json dumped the pydantic metas to plain dicts
rec = json.loads(next((run / "records").glob("*image*.json")).read_text())
for imap in rec.get("importance_maps", []):
    imap["components_meta"] = ComponentsMeta(**imap["components_meta"])
    imap["safety_meta"]     = SafetyMeta(**imap["safety_meta"])
rec["instrument_params"] = live
rec.setdefault("scan_index", 0)

state = {
    "experiment_tasks":   rec.get("experiment_tasks", []),
    "experiment_context": rec.get("experiment_context", ""),
    "instrument_state":   live,
    "scanner_calibrations": None,          # -> move_tip uses the self-anchoring path
    "decision_records":   [decision],
    "experimental_records": [rec],
    "pending": {"kind": "loop", "decision_index": int(dec.name.split("_")[1])},
}
print(f"criteria: {[n for m in rec['importance_maps'] for n in m['components_meta'].names]}")
print(f"frame   : {live.scan_settings.scan_size_m*1e6:.2f} um @ {live.scan_settings.pixels} px")
print(f"waveform: {live.loop_settings}")
print(new_run("loop_chain_test"))

20260818_161804_pick_point_single_1 | decision_01 | pfm_signal_quality/max
criteria: ['domain_interior', 'domain_wall_proximity', 'pfm_signal_quality']
frame   : 10.00 um @ 256 px
waveform: v_dc_max_v=5.0 frequency_hz=0.30048078 var0_phase=0.0 var1_pulsetime_s=0.050000001 n_cycles=2
C:\Users\Asylum User\Documents\GitHub\SPM_agent_simple\src\runs\20260818_165607_loop_chain_test


In [3]:
# [2] pick + plan — deterministic pick, then the waveform. No instrument motion.
from spm_agent.config import PICK_BORDER_PX
from spm_agent.nodes.loop_pick_point_node import loop_pick_point_node
from spm_agent.nodes.loop_plan_node import loop_plan_node

state = {**state, **(await loop_pick_point_node(state))}
row, col = state["pending"]["pixel_yx"]
N = live.scan_settings.pixels
print(f"   edge distance: {min(row, col, N-1-row, N-1-col)} px   (PICK_BORDER_PX={PICK_BORDER_PX})")

state = {**state, **(await loop_plan_node(state))}
print(state["pending"]["params"].loop_settings)

[pick] pfm_signal_quality/max -> px(21,235) | phi 0.9548, safety 1.0 | previously []
   edge distance: 20 px   (PICK_BORDER_PX=5)
[loop_plan] first loop — operator waveform adopted (v_dc_max 5.00 V, 2 cycles)
v_dc_max_v=5.0 frequency_hz=0.30048078 var0_phase=0.0 var1_pulsetime_s=0.050000001 n_cycles=2


In [4]:
# [3] MOVES THE TIP — check the target above before running
from spm_agent.nodes.move_tip_node import move_tip_node

await move_tip_node(state)

[move_tip] px(21,235) -> (+4.1992, +4.1602) um on a 10.00 um frame (39.1 nm/px)
[move_tip] achieved (+4.1992, +4.1602) um, error 0.0 nm, feedback on


{}

In [11]:
# [4] MEASURES — writes an .ibw
from spm_agent.nodes.loop_run_node import loop_run_node

state = {**state, **(await loop_run_node(state))}
print("file:", state["pending"]["path"])

[run_loop] starting — v_dc_max 5.00 V, 0.300 Hz, 2 cycles, pulse 50.0 ms (~7 s); changes=none
[run_loop] payload keys: ['loop_frequency', 'n_cycles', 'path', 'v_dc_max', 'var0_loop_phase', 'var1_pulsetime_s']
[run_loop] done -> C:\Users\Asylum User\Documents\Asylum Research Data\260816\PLZT_0025.ibw


KeyError: 'path'

In [9]:
print("file:", state["pending"]["file_path"])

KeyError: 'file_path'